In [14]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
from pathlib import Path
import numpy as np
import math
from collections import defaultdict
from itertools import combinations

In [15]:
#programme_courses = pd.read_csv("Cleaned_Data/Cleaned Programme-Courses.csv")
rooms = pd.read_csv("Cleaned_Data/Cleaned Rooms and Room Types .csv")
EMR = pd.read_csv("Cleaned_Data/Cleaned Event Module Room.csv")
EMR["Event Size"] = pd.to_numeric(EMR["Event Size"], errors="coerce").fillna(0)
#SPME = pd.read_csv("2024-5 Student Programme Module Event.csv")

In [16]:
rooms_toy = rooms[rooms["Campus"] == "Kings Buildings"]
rooms_toy = rooms_toy[["Id", "Capacity", "Building.1", "Specialist room type"]]
#rooms_toy.loc[len(rooms_toy)] = ["0787_00_XX", 1000, "Sheep Shed"]
print(rooms_toy)
print("rows:", len(rooms_toy))

                  Id  Capacity             Building.1 Specialist room type
439  0616_00_00.01-E      86.0                 Alrick     General Teaching
440  0616_01_01.03-E      80.0                 Alrick  Computer Laboratory
441  0616_02_02.02-E      72.0                 Alrick     General Teaching
442   0640_00_G.0070     313.0          Ashworth Labs     General Teaching
443   0640_00_G.0080     116.0          Ashworth Labs           Laboratory
..               ...       ...                    ...                  ...
677    0621_01_1.304       NaN                   Crew         Meeting Room
678     0648_00_G.02       NaN    Roger Land Building         Meeting Room
679     0648_00_G.13       NaN    Roger Land Building         Meeting Room
680     0648_01_1.55       NaN    Roger Land Building         Meeting Room
681     0670_01_1.07       NaN  Waddington Building 1         Meeting Room

[121 rows x 4 columns]
rows: 121


In [17]:
rooms_toy["Specialist room type"] = rooms_toy["Specialist room type"].replace(
    "Teaching Studio",
    "General Teaching"
)


rooms_toy = rooms_toy[["Id", "Capacity", "Building.1", "Specialist room type"]]

print(rooms_toy)
print("rows:", len(rooms_toy))


                  Id  Capacity             Building.1 Specialist room type
439  0616_00_00.01-E      86.0                 Alrick     General Teaching
440  0616_01_01.03-E      80.0                 Alrick  Computer Laboratory
441  0616_02_02.02-E      72.0                 Alrick     General Teaching
442   0640_00_G.0070     313.0          Ashworth Labs     General Teaching
443   0640_00_G.0080     116.0          Ashworth Labs           Laboratory
..               ...       ...                    ...                  ...
677    0621_01_1.304       NaN                   Crew         Meeting Room
678     0648_00_G.02       NaN    Roger Land Building         Meeting Room
679     0648_00_G.13       NaN    Roger Land Building         Meeting Room
680     0648_01_1.55       NaN    Roger Land Building         Meeting Room
681     0670_01_1.07       NaN  Waddington Building 1         Meeting Room

[121 rows x 4 columns]
rows: 121


In [18]:
# Rooms in Kings Buildings with missing capacity
nan_rooms = rooms_toy[rooms_toy["Capacity"].isna()]["Id"]

# Numeric conversions
emr_tmp = EMR.copy()
emr_tmp["EventSize_num"] = pd.to_numeric(emr_tmp["Event Size"], errors="coerce")
emr_tmp["Repeat_Count_num"] = pd.to_numeric(emr_tmp["Repeat_Count"], errors="coerce")

# Keep only non-repeated events
emr_tmp["Repeat_Count_num"] = emr_tmp["Repeat_Count_num"].fillna(1)
emr_nonrep = emr_tmp[emr_tmp["Repeat_Count_num"] <= 1]

# Largest event size per room, restricted to the NaN-capacity rooms
largest_by_room = (
    emr_nonrep[emr_nonrep["Room"].isin(nan_rooms)]
    .groupby("Room", dropna=True)["EventSize_num"]
    .max()
    .rename("largest_event_size_non_repeated")
)

result = (
    rooms_toy[rooms_toy["Capacity"].isna()][["Id", "Building.1", "Specialist room type"]]
    .merge(largest_by_room, left_on="Id", right_index=True, how="left")
    .sort_values("Id")
)
print(result)

rooms_toy = rooms_toy.copy()
rooms_toy["Capacity"] = pd.to_numeric(rooms_toy["Capacity"], errors="coerce")
rooms_toy.loc[rooms_toy["Capacity"].isna(), "Capacity"] = (
    rooms_toy.loc[rooms_toy["Capacity"].isna(), "Id"]
    .map(largest_by_room)
    .mul(1.0)
    .apply(lambda x: pd.NA if pd.isna(x) else int(pd.Series([x]).round(0).iloc[0]))
)

print(rooms_toy[rooms_toy["Id"].isin(nan_rooms)])


                Id             Building.1 Specialist room type  \
676  0621_01_1.302                   Crew         Meeting Room   
677  0621_01_1.304                   Crew         Meeting Room   
678   0648_00_G.02    Roger Land Building         Meeting Room   
679   0648_00_G.13    Roger Land Building         Meeting Room   
680   0648_01_1.55    Roger Land Building         Meeting Room   
681   0670_01_1.07  Waddington Building 1         Meeting Room   

     largest_event_size_non_repeated  
676                             40.0  
677                              6.0  
678                             40.0  
679                             10.0  
680                             12.0  
681                             12.0  
                Id  Capacity             Building.1 Specialist room type
676  0621_01_1.302      40.0                   Crew         Meeting Room
677  0621_01_1.304       6.0                   Crew         Meeting Room
678   0648_00_G.02      40.0    Roger Land Bu

In [19]:
print(rooms_toy["Specialist room type"].value_counts())

Specialist room type
General Teaching       62
Laboratory             36
Computer Laboratory    13
Meeting Room           10
Name: count, dtype: int64


In [20]:
csv_files = [
    "KB_chem_phys_30_9to6.csv",
    "KB_eng_geo_bio_30_9to6.csv",
    "KB_math_30_9to6.csv",
]

slot_minutes = 30

dfs = [pd.read_csv(f) for f in csv_files]
assign_df = pd.concat(dfs, ignore_index=True)

print(assign_df.shape)
display(assign_df.head())


(15010, 6)


,Event ID,Time Assignment,Event Size,Room Type,Online Status,Campus
0,E:MXQUIA3ANO-00001,"Week 9, Friday, 09:00",60.0,General Teaching,Not online,Kings Buildings
1,E:NUBX7AZ7CB,"Week 9, Friday, 09:00",140.0,General Teaching,Not online,Kings Buildings
2,E:OD63917BLY,"Week 9, Friday, 09:00",30.0,General Teaching,Not online,Kings Buildings
3,E:OM3A3FTRKS-00002,"Week 9, Friday, 09:00",300.0,General Teaching,Not online,Kings Buildings
4,E:OR6WOD6AKH,"Week 9, Friday, 09:00",40.0,General Teaching,Not online,Kings Buildings


In [21]:
### Room Assignment Input

# Keep only in-person events that need a room
assign_df = assign_df[
    (assign_df["Online Status"] == "Not online") &
    assign_df["Room Type"].notna() &
    (assign_df["Room Type"] != "No room required")
].copy()

# Parse "Week 9, Friday, 09:00"
parts = assign_df["Time Assignment"].str.extract(r"Week (\d+), ([^,]+), (\d{2}):(\d{2})")
assign_df["Week"] = parts[0].astype(int)
assign_df["Day"] = parts[1]
assign_df["Hour"] = parts[2].astype(int)
assign_df["Minute"] = parts[3].astype(int)

if slot_minutes == 30:
    assign_df["StartSlot"] = ((assign_df["Hour"] - 9) * 2 + (assign_df["Minute"] // 30) + 1).astype(int)
else:
    assign_df["StartSlot"] = ((assign_df["Hour"] - 9) + 1).astype(int)

# Bring in event durations
event_durations = (
    EMR[["Event ID", "Duration (minutes)"]]
    .dropna(subset=["Event ID", "Duration (minutes)"])
    .drop_duplicates(subset=["Event ID"])
)

assign_df = assign_df.merge(event_durations, on="Event ID", how="left")
assign_df["DurSlots"] = assign_df["Duration (minutes)"].apply(lambda x: int(math.ceil(x / slot_minutes)))

# Unique scheduled occurrence key
assign_df["OccKey"] = list(zip(
    assign_df["Event ID"],
    assign_df["Week"],
    assign_df["Day"],
    assign_df["StartSlot"]
))

display(assign_df.head())
print("Events to room-assign:", len(assign_df))


,Event ID,Time Assignment,Event Size,Room Type,Online Status,Campus,Week,Day,Hour,Minute,StartSlot,Duration (minutes),DurSlots,OccKey
0,E:MXQUIA3ANO-00001,"Week 9, Friday, 09:00",60.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:MXQUIA3ANO-00001, 9, Friday, 1)"
1,E:NUBX7AZ7CB,"Week 9, Friday, 09:00",140.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:NUBX7AZ7CB, 9, Friday, 1)"
2,E:OD63917BLY,"Week 9, Friday, 09:00",30.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:OD63917BLY, 9, Friday, 1)"
3,E:OM3A3FTRKS-00002,"Week 9, Friday, 09:00",300.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:OM3A3FTRKS-00002, 9, Friday, 1)"
4,E:OR6WOD6AKH,"Week 9, Friday, 09:00",40.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:OR6WOD6AKH, 9, Friday, 1)"


Events to room-assign: 14889


In [22]:
### Room Candidates and Penalties

rooms_assign = rooms_toy.copy()

def compatible_room(required_type, room_type):
    if pd.isna(required_type) or pd.isna(room_type):
        return False
    if required_type == "General Teaching":
        return True
    if required_type == "Teaching Studio":
        return room_type in {"Teaching Studio", "General Teaching"}
    return room_type == required_type

occurrences = assign_df["OccKey"].tolist()
rooms_list = rooms_assign["Id"].tolist()

occ_req_type = assign_df.set_index("OccKey")["Room Type"].to_dict()
occ_size = assign_df.set_index("OccKey")["Event Size"].to_dict()
occ_week = assign_df.set_index("OccKey")["Week"].to_dict()
occ_day = assign_df.set_index("OccKey")["Day"].to_dict()
occ_start = assign_df.set_index("OccKey")["StartSlot"].to_dict()
occ_len = assign_df.set_index("OccKey")["DurSlots"].to_dict()
occ_event = assign_df.set_index("OccKey")["Event ID"].to_dict()

room_type = rooms_assign.set_index("Id")["Specialist room type"].to_dict()
room_cap = rooms_assign.set_index("Id")["Capacity"].to_dict()

candidate_pairs = []
linear_penalty = {}
quadratic_penalty = {}

for occ in occurrences:
    req = occ_req_type[occ]
    size = float(occ_size[occ])

    for r in rooms_list:
        if compatible_room(req, room_type[r]):
            cap = room_cap[r]
            shortfall = 0 if pd.isna(cap) else max(0.0, size - float(cap))

            candidate_pairs.append((occ, r))
            linear_penalty[(occ, r)] = shortfall
            quadratic_penalty[(occ, r)] = shortfall ** 2

print("Candidate occurrence-room pairs:", len(candidate_pairs))


Candidate occurrence-room pairs: 1565642


In [23]:
occupancy = defaultdict(list)

for occ in occurrences:
    w = occ_week[occ]
    d = occ_day[occ]
    s0 = occ_start[occ]
    L = occ_len[occ]

    for t in range(s0, s0 + L):
        occupancy[(w, d, t)].append(occ)

occ_index = list(occupancy.keys())
print("Occupied time indices:", len(occ_index))


Occupied time indices: 990


In [24]:
m2 = pyo.ConcreteModel()

m2.O = pyo.Set(initialize=occurrences, dimen=4)
m2.R = pyo.Set(initialize=rooms_list)
m2.A = pyo.Set(dimen=5, initialize=[(e, w, d, s, r) for ((e, w, d, s), r) in candidate_pairs])
m2.Occ = pyo.Set(dimen=3, initialize=occ_index)

m2.z = pyo.Var(m2.A, domain=pyo.Binary)

# Each occurrence gets exactly one room
def one_room_rule(m, e, w, d, s):
    return sum(m.z[e, w, d, s, r] for r in m.R if (e, w, d, s, r) in m.A) == 1
m2.one_room = pyo.Constraint(m2.O, rule=one_room_rule)

# No room can host two occurrences in the same occupied slot
def no_clash_rule(m, w, d, t, r):
    occs = occupancy[(w, d, t)]
    feasible = [(e, ww, dd, ss, r) for (e, ww, dd, ss) in occs if (e, ww, dd, ss, r) in m.A]
    if not feasible:
        return pyo.Constraint.Skip
    return sum(m.z[e, ww, dd, ss, r] for (e, ww, dd, ss, r) in feasible) <= 1
m2.no_clash = pyo.Constraint(m2.Occ, m2.R, rule=no_clash_rule)


In [25]:
m2.obj = pyo.Objective(
    expr=sum(quadratic_penalty[((e, w, d, s), r)] * m2.z[e, w, d, s, r] for (e, w, d, s, r) in m2.A),
    sense=pyo.minimize
)

In [26]:
solver = pyo.SolverFactory("gurobi")

#solver.options["NonConvex"] = 2
#solver.options["MIPFocus"] = 1
#solver.options["Heuristics"] = 0.4
#solver.options["NoRelHeurTime"] = 1000
solver.options["TimeLimit"] = 100

res_room = solver.solve(m2, tee=True)

print("Room assignment termination:", res_room.solver.termination_condition)
print("Room assignment status:", res_room.solver.status)

linear_total = sum(
    linear_penalty[((e, w, d, s), r)] * pyo.value(m2.z[e, w, d, s, r])
    for (e, w, d, s, r) in m2.A
)

quadratic_total = sum(
    quadratic_penalty[((e, w, d, s), r)] * pyo.value(m2.z[e, w, d, s, r])
    for (e, w, d, s, r) in m2.A
)

print("Linear penalty:", linear_total)
print("Quadratic penalty:", quadratic_total)


Read LP format model from file C:\Users\asher\AppData\Local\Temp\tmp7wkxjb55.pyomo.lp
Reading time = 11.47 seconds
x1: 134679 rows, 1565642 columns, 6205455 nonzeros
Set parameter TimeLimit to value 100
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  100

Optimize a model with 134679 rows, 1565642 columns and 6205455 nonzeros (Min)
Model fingerprint: 0xbbd130dd
Model has 623649 linear objective coefficients
Variable types: 0 continuous, 1565642 integer (1565642 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 6.911944e+07
Presolve removed 0 rows and 0 columns (presolve time = 5s)...
Presolve removed 1306

In [27]:
assigned_rooms = []
for (e, w, d, s, r) in m2.A:
    if pyo.value(m2.z[e, w, d, s, r]) > 0.5:
        assigned_rooms.append((e, w, d, s, r, room_cap[r], room_type[r]))

assigned_rooms_df = pd.DataFrame(
    assigned_rooms,
    columns=["Event ID", "Week", "Day", "StartSlot", "Assigned Room", "Room Capacity", "Assigned Room Type"]
)

final_room_output = assign_df.merge(
    assigned_rooms_df,
    on=["Event ID", "Week", "Day", "StartSlot"],
    how="left"
)

display(final_room_output.head(20))

final_room_output.to_csv("KB_Room_Allocation_30_9to6.csv", index=False)
print("Saved: KB_Room_Allocation_30_9to6.csv")


,Event ID,Time Assignment,Event Size,Room Type,Online Status,Campus,Week,Day,Hour,Minute,StartSlot,Duration (minutes),DurSlots,OccKey,Assigned Room,Room Capacity,Assigned Room Type
0,E:MXQUIA3ANO-00001,"Week 9, Friday, 09:00",60.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:MXQUIA3ANO-00001, 9, Friday, 1)",0633_01_1.201,135.0,General Teaching
1,E:NUBX7AZ7CB,"Week 9, Friday, 09:00",140.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:NUBX7AZ7CB, 9, Friday, 1)",0639_02_2.07,150.0,General Teaching
2,E:OD63917BLY,"Week 9, Friday, 09:00",30.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:OD63917BLY, 9, Friday, 1)",0640_00_G.0085_G.0088,64.0,Laboratory
3,E:OM3A3FTRKS-00002,"Week 9, Friday, 09:00",300.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:OM3A3FTRKS-00002, 9, Friday, 1)",0639_-1_B.01,300.0,General Teaching
4,E:OR6WOD6AKH,"Week 9, Friday, 09:00",40.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:OR6WOD6AKH, 9, Friday, 1)",0613_06_6.6307,45.0,Laboratory
5,E:QGSC7NW2UU,"Week 9, Friday, 09:00",180.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:QGSC7NW2UU, 9, Friday, 1)",0632_00_G.024,230.0,General Teaching
6,E:R5JC23OERW-00001,"Week 9, Friday, 09:00",200.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:R5JC23OERW-00001, 9, Friday, 1)",0640_00_G.0070,313.0,General Teaching
7,E:VBCLVPOK6O,"Week 9, Friday, 09:00",65.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:VBCLVPOK6O, 9, Friday, 1)",0616_00_00.01-E,86.0,General Teaching
8,E:WI3SCGO928,"Week 9, Friday, 09:00",130.0,General Teaching,Not online,Kings Buildings,9,Friday,9,0,1,50,2,"(E:WI3SCGO928, 9, Friday, 1)",0612_01_1.02,372.0,General Teaching
9,E:DWVATU9VFQ,"Week 9, Friday, 09:30",15.0,General Teaching,Not online,Kings Buildings,9,Friday,9,30,2,110,4,"(E:DWVATU9VFQ, 9, Friday, 2)",0639_00_G.02,112.0,General Teaching


Saved: KB_Room_Allocation_30_9to6.csv
